# M2: CLV 가중 전체 가격거리 표현 (Dunnhumby, seed 42)

기존 LightGCN ID 64차원에 `-q_C(u)(p_i-p_u)^2`와 순위·BPR 관점에서 정확히 같은 2차원 내적기저만 layer-0에 추가합니다. 관계 블록, N/V 독립 점수, 카테고리 내 가격, 상품 ID 보조사영은 사용하지 않습니다. 동일 초기화의 `rho=0` 대조군과 `rho=0.05` M2를 고정 100 epoch로 비교하며, 최종 test와 holdout은 생성하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/jung-un/clv-m2-lightgcn-runner.git'
SOURCE_COMMIT = 'a88421008d9f4a1ae2ce24968d8677dfd390e971'
REPO_DIR = '/content/clv-m2-lightgcn-runner'
if len(SOURCE_COMMIT) != 40:
    raise RuntimeError('검토된 소스 커밋을 SOURCE_COMMIT에 고정해야 합니다')
!if [ -d {REPO_DIR}/.git ]; then git -C {REPO_DIR} fetch origin; else git clone {REPO_URL} {REPO_DIR}; fi
!git -C {REPO_DIR} checkout {SOURCE_COMMIT}
%cd {REPO_DIR}


In [ ]:
import json
import torch
from lightgcn_clv_weighted_price_distance import (
    configure_price_distance_run,
    preflight_summary,
    run_price_distance_screen,
)

assert torch.cuda.is_available(), '런타임 유형에서 GPU를 선택하세요.'
cfg = configure_price_distance_run()
print(json.dumps(preflight_summary(cfg), ensure_ascii=False, indent=2))


In [ ]:
result_df = run_price_distance_screen(cfg)


In [ ]:
from IPython.display import display
import pandas as pd

print('1) 절대지표: M1, rho=0, 최소 가격거리 M2, 공동학습 ID-only')
display(result_df)
print('2) rho=0 및 공동학습 ID-only 대비 전체·CLV 구간 성과')
display(pd.DataFrame(result_df.attrs['comparison']))
print('3) CLV 구간별 Top-10 변경')
display(pd.DataFrame(result_df.attrs['top10_overlap']))
print('4) 실제 점수 영향력과 학습된 가격 scale')
display(pd.DataFrame(result_df.attrs['score_diagnostics']))
print('5) 사전 판정 규칙 결과')
print(json.dumps(result_df.attrs['screening_reading'], ensure_ascii=False, indent=2))
print('6) 저장 파일')
print(json.dumps(result_df.attrs['result_paths'], ensure_ascii=False, indent=2))
